In [40]:
!pip install pandas numpy scikit-learn matplotlib pyarrow -q


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\9901359\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [41]:
import pandas as pd
import numpy as np
import os, json, csv, time, logging, random
from datetime import time, datetime, timedelta
import sqlite3

random.seed(42)
np.random.seed(42)
logging.basicConfig(level = logging.INFO, format = "%(asctime)s [%(levelname)s] %(message)s")
print("Environment setup is done")

Environment setup is done


In [56]:
import pandas
print(pandas.__file__)


C:\Users\9901359\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pandas\__init__.py


In [42]:
# generating the dataset

api_records = []

for page in range(1, 6):  # 5 pages
    for i in range(100):  # 100 per page
        idx = (page-1)*100 + i
        api_records.append({
            'id': idx+1,
            'name': f'Supplier_{idx+1}',
            'country': random.choice(['US','UK','DE','JP','IN','BR']),
            'category': random.choice(['Electronics','Textiles','Chemicals','Food','Metals']),
            'annual_revenue': round(random.uniform(100000, 50000000), 2),
            'rating': round(random.uniform(1, 5), 1),
            'is_active': random.choice([True, True, True, False])
        })
df = pd.DataFrame(api_records)
df.head()

,id,name,country,category,annual_revenue,rating,is_active
0,1,Supplier_1,BR,Electronics,1348036.69,2.1,True
1,2,Supplier_2,UK,Electronics,33867304.42,4.6,True
2,3,Supplier_3,IN,Food,1685955.71,1.4,True
3,4,Supplier_4,IN,Metals,1424144.89,1.8,False
4,5,Supplier_5,UK,Food,29504357.63,4.2,True


In [43]:
# Dataprofiling and quality check

print('=' * 60)
print('DATA PROFILE: JSON Parsing and Authentication')
print('=' * 60)
print(f'\nShape: {df.shape}')
print(f'\nColumn Types:\n{df.dtypes}')
print(f'\nNull Counts:\n{df.isnull().sum()}')
print(f'\nDuplicate Rows: {df.duplicated().sum()}')
print(f'\nNumeric Summary:\n{df.describe()}')
print(f'\nUnique Values:')
for col in df.columns:
    print(f'  {col}: {df[col].nunique()} unique')
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

DATA PROFILE: JSON Parsing and Authentication

Shape: (500, 7)

Column Types:
id                  int64
name                  str
country               str
category              str
annual_revenue    float64
rating            float64
is_active            bool
dtype: object

Null Counts:
id                0
name              0
country           0
category          0
annual_revenue    0
rating            0
is_active         0
dtype: int64

Duplicate Rows: 0

Numeric Summary:
               id  annual_revenue      rating
count  500.000000    5.000000e+02  500.000000
mean   250.500000    2.489543e+07    3.020400
std    144.481833    1.446659e+07    1.153414
min      1.000000    4.046789e+05    1.000000
25%    125.750000    1.277595e+07    2.100000
50%    250.500000    2.489018e+07    3.000000
75%    375.250000    3.779579e+07    4.000000
max    500.000000    4.997979e+07    5.000000

Unique Values:
  id: 500 unique
  name: 500 unique
  country: 6 unique
  category: 5 unique
  annual_revenu

In [44]:
# core implementation - json parsing
import json
response = {
    'status': 'success',
    'data': {
        'supplier': {
            'id': 42,
            'name': 'TechParts Global',
            'country': 'Germany',
            'contacts': {
                'primary': {'name': 'Hans Mueller', 'email': 'hans@techparts.de', 'phone': '+49-555-0101'},
                'billing': {'name': 'Eva Schmidt', 'email': 'billing@techparts.de'}
            },
            'products': [
                {'id': 'P001', 'name': 'Capacitor 100uF', 'unit_price': 0.45, 'moq': 1000},
                {'id': 'P002', 'name': 'Resistor 10k', 'unit_price': 0.12, 'moq': 5000},
                {'id': 'P003', 'name': 'LED 5mm Red', 'unit_price': 0.08, 'moq': 10000}
            ],
            'certifications': ['ISO9001', 'ISO14001', 'RoHS'],
            'metadata': {'last_audit': '2024-06-15', 'tier': 'Gold'}
        }
    },
    'pagination': {'page': 1, 'total_pages': 1}
}


#safe navigation
supplier = response.get('data',{}).get('supplier',{})
name = supplier.get('name','Unknown')
country = supplier.get('country','Unknown')
email = supplier.get('contacts', {}).get('primary', {}).get('email', 'N/A')
tier = supplier.get('metadata',{}).get('tier',{})

print(f'Supplier: {name} | Country: {country} | Email: {email} | Tier: {tier}')

products = supplier.get('products',[])
for p in products:
    print(f'  {p["id"]}: {p["name"]} — ${p["unit_price"]:.2f} (MOQ: {p["moq"]})')


required_fields = ['id', 'name', 'country', 'contacts', 'products']
missing = []
for i in required_fields:
    if i not in supplier:
        missing.append(i)
if missing:
    print(f'  FAIL: Missing fields: {missing}')
else:
    print(f'  PASS: All {len(required_fields)} required fields present')


# validating primary in the contacts
has_primary_contact = 'primary' in supplier.get('contacts', {})
has_products = len(supplier.get('products',[]))>0
print(has_primary_contact)
print(has_products)

# flattenning records from json
# APIs often return nested JSON, but for analysis you want a flat table (like a DataFrame).
# The code builds flat_records where each product is combined with supplier info

flat_records = []

for p in products:
    flat_records.append({
        'supplier_id': supplier['id'],
        'supplier_name': supplier['name'],
        'supplier_country': supplier['country'],
        'contact_email': email,
        'product_id': p['id'],
        'product_name': p['name'],
        'unit_price': p['unit_price'],
        'moq': p['moq'],
        'tier': tier
    })

df_flat = pd.DataFrame(flat_records)
print(df_flat)

# --- Parse multiple API responses (batch) ---
print('\n=== 5. Batch JSON Processing ===')
# Generate 50 mock API responses
random.seed(42)
api_responses = []
for i in range(50):
    api_responses.append({
        'id': i + 1,
        'name': f'Supplier_{i+1}',
        'country': random.choice(['US', 'UK', 'DE', 'JP', 'IN']),
        'contacts': {
            'primary': {'email': f'contact{i+1}@supplier.com'}
        },
        'products': [
            {'name': f'Prod_{j}', 'price': round(random.uniform(1, 100), 2)}
            for j in range(random.randint(1, 5))
        ],
        'rating': round(random.uniform(1, 5), 1)
    })

# Flatten all responses
all_flat = []
for resp in api_responses:
    for prod in resp['products']:
        all_flat.append({
            'supplier_id': resp['id'],
            'supplier_name': resp['name'],
            'country': resp['country'],
            'email': resp['contacts']['primary']['email'],
            'product': prod['name'],
            'price': prod['price'],
            'rating': resp['rating']
        })
df_batch = pd.DataFrame(all_flat)
print(f'Flattened {len(api_responses)} responses into {len(df_batch)} product rows')
print(f'Columns: {list(df_batch.columns)}')
print(df_batch.head())

# --- JSON serialization / deserialization ---
print('\n=== 6. JSON Serialization ===')
sample = df_batch.head(3).to_dict(orient='records')
json_str = json.dumps(sample, indent=2)
print(f'Serialized to JSON string ({len(json_str)} chars):')
print(json_str[:200] + '...')

# Parse it back
parsed = json.loads(json_str)
print(f'Deserialized back: {len(parsed)} records')

print('\n-- JSON Parsing and Authentication implementation complete')


Supplier: TechParts Global | Country: Germany | Email: hans@techparts.de | Tier: Gold
  P001: Capacitor 100uF — $0.45 (MOQ: 1000)
  P002: Resistor 10k — $0.12 (MOQ: 5000)
  P003: LED 5mm Red — $0.08 (MOQ: 10000)
  PASS: All 5 required fields present
True
True
   supplier_id     supplier_name supplier_country      contact_email  \
0           42  TechParts Global          Germany  hans@techparts.de   
1           42  TechParts Global          Germany  hans@techparts.de   
2           42  TechParts Global          Germany  hans@techparts.de   

  product_id     product_name  unit_price    moq  tier  
0       P001  Capacitor 100uF        0.45   1000  Gold  
1       P002     Resistor 10k        0.12   5000  Gold  
2       P003      LED 5mm Red        0.08  10000  Gold  

=== 5. Batch JSON Processing ===
Flattened 50 responses into 132 product rows
Columns: ['supplier_id', 'supplier_name', 'country', 'email', 'product', 'price', 'rating']
   supplier_id supplier_name country                

In [45]:
# Cell 5: Results Analysis
print('=' * 60)
print('RESULTS: JSON Parsing and Authentication')
print('=' * 60)
print(f'Input rows: {len(df)}')
print(f'Processing complete')

# Display key metrics
print(f'\nTop 5 rows of results:')
print(df.head())

RESULTS: JSON Parsing and Authentication
Input rows: 500
Processing complete

Top 5 rows of results:
   id        name country     category  annual_revenue  rating  is_active
0   1  Supplier_1      BR  Electronics      1348036.69     2.1       True
1   2  Supplier_2      UK  Electronics     33867304.42     4.6       True
2   3  Supplier_3      IN         Food      1685955.71     1.4       True
3   4  Supplier_4      IN       Metals      1424144.89     1.8      False
4   5  Supplier_5      UK         Food     29504357.63     4.2       True


In [46]:
# Cell 6: Self-Check — JSON Parsing
# Run this cell to verify your work before clicking "Run Tests"
print('=' * 50)
print('SELF-CHECK — JSON Parsing')
print('=' * 50)

checks = {
    'DataFrame exists and is not empty': len(df) > 0,
    'Has at least 2 columns': len(df.columns) >= 2,
    'No fully-null columns': df.isnull().mean().max() < 0.5,
    'Has at least 10 rows': len(df) >= 10,
    'Data types look valid': df.dtypes is not None,
}

passed = sum(1 for v in checks.values() if v)
for name, ok in checks.items():
    print(f'  {"PASS" if ok else "FAIL"}: {name}')

print(f'\n{passed}/{len(checks)} self-checks passed')
if passed == len(checks):
    print('\nAll good! Click "Run Tests" to submit for official validation.')
else:
    print('\nSome checks failed. Fix your code, then click "Run Tests".')

SELF-CHECK — JSON Parsing
  PASS: DataFrame exists and is not empty
  PASS: Has at least 2 columns
  PASS: No fully-null columns
  PASS: Has at least 10 rows
  PASS: Data types look valid

5/5 self-checks passed

All good! Click "Run Tests" to submit for official validation.


In [47]:
import os
def create_file(filename, content = "", overwrite = False):
    try:
        if not filename or not isinstance(filename,str):
            raise ValueError("Invalid filename provided")
        curr = os.getcwd()
        print(curr)
        filepath = os.path.join(curr,filename)
        if os.path.exists(filepath) and not overwrite:
            print(f"Error: File '{filename}' already exists. Use overwrite=True to replace it.")
            return
        with open(filepath, 'w', encoding = 'utf-8') as file:
            file.write(content)
    except Exception as e:
        print("Error occurred",e)
#create_file(".env","client_id=abc123 client_secret=xyz456 base_url=https://api.example.com", overwrite = False)
create_file(
    ".env",
    "client_id=abc123\nclient_secret=xyz456\nbase_url=https://api.example.com",
    overwrite=True
)
        
print("file created")

c:\Users\9901359\Downloads\DATA ENGINEERING LEARNLYTICS\DAY 12 JSON PARSING
file created


In [48]:
import os
import time
import requests
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
client_id = os.getenv("client_id")
client_secret = os.getenv("client_secret")
base_url = os.getenv("base_url")

if not client_id or not client_secret or not base_url:
    raise ValueError("Missing credentials in environment file")

class AuthError(Exception):
    pass

class APIClient:
    def __init__(self, client_id, client_secret, base_url):
        self.client_id = client_id
        self.client_secret = client_secret
        self.base_url = base_url
        self.token = None
        self.token_expiry = 0

    def _get_token(self):
        url = f"{self.base_url}/oauth/token"
        payload = {
            "client_id": self.client_id,
            "client_secret": self.client_secret,
            "grant_type": "client_credentials"
        }
        resp = requests.post(url, data=payload)
        if resp.status_code != 200:
            raise AuthError("Failed to get token")
        data = resp.json()
        self.token = data["access_token"]
        self.token_expiry = time.time() + data.get("expires_in", 3600)

    def _ensure_token(self):
        if self.token is None or time.time() > self.token_expiry - 60:
            self._get_token()

    def get(self, endpoint, params=None):
        self._ensure_token()
        headers = {"Authorization": f"Bearer {self.token}"}
        resp = requests.get(f"{self.base_url}{endpoint}", headers=headers, params=params)
        if resp.status_code == 401:
            self._get_token()
            headers = {"Authorization": f"Bearer {self.token}"}
            resp = requests.get(f"{self.base_url}{endpoint}", headers=headers, params=params)
            if resp.status_code == 401:
                raise AuthError("Unauthorized after retry")
        return resp.json()

    def post(self, endpoint, payload=None):
        self._ensure_token()
        headers = {"Authorization": f"Bearer {self.token}"}
        resp = requests.post(f"{self.base_url}{endpoint}", headers=headers, json=payload)
        if resp.status_code == 401:
            self._get_token()
            headers = {"Authorization": f"Bearer {self.token}"}
            resp = requests.post(f"{self.base_url}{endpoint}", headers=headers, json=payload)
            if resp.status_code == 401:
                raise AuthError("Unauthorized after retry")
        return resp.json()


In [51]:
import inspect
import requests
print(inspect.getsource(json.loads))

def loads(s, *, cls=None, object_hook=None, parse_float=None,
        parse_int=None, parse_constant=None, object_pairs_hook=None, **kw):
    """Deserialize ``s`` (a ``str``, ``bytes`` or ``bytearray`` instance
    containing a JSON document) to a Python object.

    ``object_hook`` is an optional function that will be called with the
    result of any object literal decode (a ``dict``). The return value of
    ``object_hook`` will be used instead of the ``dict``. This feature
    can be used to implement custom decoders (e.g. JSON-RPC class hinting).

    ``object_pairs_hook`` is an optional function that will be called with the
    result of any object literal decoded with an ordered list of pairs.  The
    return value of ``object_pairs_hook`` will be used instead of the ``dict``.
    This feature can be used to implement custom decoders.  If ``object_hook``
    is also defined, the ``object_pairs_hook`` takes priority.

    ``parse_float``, if specified, will be called with the str